# Part 1. Equation of a Slime

How many late days are you using for this assignment? 0


In [42]:
# Imports section
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier



## 1. Loading the dataset

In [43]:
# Using pandas load the dataset
# Output the first 15 rows of the data
# Display a summary of the table information (number of datapoints, etc.)
df = pd.read_csv('science_data_large.csv')
print(df.head(15))
print(df.info())


    Temperature °C  Mols KCL     Size nm^3
0              469       647  6.244743e+05
1              403       694  5.779610e+05
2              302       975  6.196847e+05
3              779       916  1.460449e+06
4              901        18  4.325726e+04
5              545       637  7.124634e+05
6              660       519  7.006960e+05
7              143       869  2.718260e+05
8               89       461  8.919803e+04
9              294       776  4.770210e+05
10             991       117  2.441771e+05
11             307       781  5.006455e+05
12             206        70  3.145200e+04
13             437       599  5.390215e+05
14             566        75  9.185271e+04
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature °C  1000 non-null   int64  
 1   Mols KCL        1000 non-null   int64  
 2   Size nm^3       1000 n

## 2. Splitting the dataset

In [44]:
# Take the pandas dataset and split it into our features (X) and label (y)
X = df.drop('Size nm^3', axis=1)  
y = df['Size nm^3']
# Use sklearn to split the features and labels into a training/test set. (90% train, 10% test)
# For grading consistency use random_state=42 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

## 3. Perform a Linear Regression

In [45]:
# Use sklearn to train a model on the training set
model = LinearRegression()
model.fit(X_train, y_train)
# Create a sample datapoint and predict the output of that sample with the trained model
sample_data_point = X_test.iloc[0:1]
predictions = model.predict(sample_data_point)
print("Prediction: ", predictions)
# Report the score for that model using the default score function property of the SKLearn model, in your own words (markdown, not code) explain what the score means
score = model.score(X_test, y_test)
print("Score: ", score)

# Extract the coefficients and intercept from the model and write an equation for your h(x) using LaTeX
coefficients = model.coef_
intercept = model.intercept_
print(f'Intercept: {intercept:.5f}')
for feature, coef in zip(X_train.columns, coefficients):
    print(f'Feature: {feature}, Coefficient: {coef:.5f}')


Prediction:  [235911.1927226]
Score:  0.8552472077276095
Intercept: -409391.47958
Feature: Temperature °C, Coefficient: 866.14641
Feature: Mols KCL, Coefficient: 1032.69507


Write the linear equation of a slime: (example equation: $E = mc^2$)

$$
h(x) = -409391.47958 + 866.14641 \cdot \text{Temperature} + 1032.69507 \cdot \text{Mols KCL}
$$

Report on score and explain meaning:

The score my model got was 0.85525.  The score is the coeffiecient of determination, or R^2, and measures how well the model explains the variance in the target variable based on the features provided.  A higher R^2 typically indicates a better fit of the model to the data, with the max value being 1.  My score of 0.85525 indicates strong predictive power and means the model captures over 3/4 the variance in the data.

## 4. Use Cross Validation

In [46]:
# Use the cross_val_score function to repeat your experiment across many shuffles of the data
# For grading consistency use n_splits=5 and random_state=42
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kf)
print(f'Cross-validation scores: {scores}')
print(f'Mean cross-validation score: {np.mean(scores)}')

# Report on their finding and their significance

Cross-validation scores: [0.86151889 0.82742341 0.87195173 0.88166206 0.85609101]
Mean cross-validation score: 0.8597294202684646


Write findings here:  My model had a mean cross-validation score of around 0.86, indicating that my model had strong performance across multiple train-test splits.  Given the consistency of all 5 scores which stayed in the 0.8 range, this model should perform well on unseen data, and captures over 80% of the variance in the data.

## 5. Using Polynomial Regression

In [47]:
# Using the PolynomialFeatures library perform another regression on an augmented dataset of degree 2
# Perform k-fold cross validation (as above)
degree = 2
pipeline = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2)), 
    ('lin_reg', LinearRegression())                         
])
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=kf)

print(f'Cross-validation scores: {scores}')
print(f'Mean cross-validation score: {np.mean(scores)}')

# Report on the metrics and output the resultant equation as you did in Part 3.

pipeline.fit(X, y)

# Extract the linear regression model from the pipeline
linear_model = pipeline.named_steps['lin_reg']

# Coefficients and intercept
coefficients = linear_model.coef_
intercept = linear_model.intercept_

# Generate feature names
poly = pipeline.named_steps['poly_features']
feature_names = poly.get_feature_names_out(X.columns)

# Output the intercept and coefficients
print("Intercept:", intercept)
for feature, coef in zip(feature_names, coefficients):
    print(f"Feature: {feature}, Coefficient: {coef:.5f}")



Cross-validation scores: [1. 1. 1. 1. 1.]
Mean cross-validation score: 1.0
Intercept: 1.657102257013321e-05
Feature: 1, Coefficient: 0.00000
Feature: Temperature °C, Coefficient: 12.00000
Feature: Mols KCL, Coefficient: -0.00000
Feature: Temperature °C^2, Coefficient: -0.00000
Feature: Temperature °C Mols KCL, Coefficient: 2.00000
Feature: Mols KCL^2, Coefficient: 0.02857


Write the polynomial equation of a slime: (example equation: $E = mc^2$)

$$
h(x) = 1.65710e^{-5} + 12 \cdot \text{Temperature} + 2 \cdot \text{Mols KCL} + 0.02857 \cdot \text{Mols KCL}^2
$$

Report on the score and interpret:

Every cross-validation score returned as 1.0, meaning that the mean score was 1.0.  My model explains 100% of the variance in the target variable, meaning that there is no error in predictions and it will perform perfectly in determining the size of a slime.

# Part 2. Chronic Kidney Disease Prediction via Classification

Create code and markdown cells as needed to perform classification and report on your results

In [48]:
# Load the dataset. Then train and evaluate the classification models.
df = pd.read_csv('ckd_feature_subset.csv') #Dataset loaded
X = df.drop('Target_ckd', axis=1)  
y = df['Target_ckd']
 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
 
#Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg_scores = cross_val_score(log_reg, X_scaled, y, cv=kf)
print(f'Logistic Regression Cross-validation scores: {log_reg_scores}')
print(f'Mean score: {np.mean(log_reg_scores)}, Standard deviation: {np.std(log_reg_scores)}\n')
 
# Support Vector Machines
svc = SVC()
svc_scores = cross_val_score(svc, X_scaled, y, cv=kf)
print(f'Support Vector Machines Cross-validation scores: {svc_scores}')
print(f'Mean score: {np.mean(svc_scores)}, Standard deviation: {np.std(svc_scores)}\n')

# k-Nearest Neighbors
knn = KNeighborsClassifier()
knn_scores = cross_val_score(knn, X_scaled, y, cv=kf)
print(f'k-Nearest Neighbors Cross-validation scores: {knn_scores}')
print(f'Mean score: {np.mean(knn_scores)}, Standard deviation: {np.std(knn_scores)}\n')

# Neural Networks
mlp = MLPClassifier(max_iter=1000)
mlp_scores = cross_val_score(mlp, X_scaled, y, cv=kf)
print(f'Neural Networks Cross-validation scores: {mlp_scores}')
print(f'Mean score: {np.mean(mlp_scores)}, Standard deviation: {np.std(mlp_scores)}')


Logistic Regression Cross-validation scores: [0.93548387 1.         0.87096774 0.96666667 0.96666667]
Mean score: 0.9479569892473119, Standard deviation: 0.043569489112761005

Support Vector Machines Cross-validation scores: [1.         1.         0.93548387 0.93333333 0.96666667]
Mean score: 0.9670967741935484, Standard deviation: 0.02934203509816868

k-Nearest Neighbors Cross-validation scores: [0.93548387 1.         0.83870968 0.86666667 0.96666667]
Mean score: 0.921505376344086, Standard deviation: 0.060429724867728095

Neural Networks Cross-validation scores: [0.96774194 1.         0.87096774 0.93333333 0.96666667]
Mean score: 0.9477419354838711, Standard deviation: 0.0437971126177301


## Results and Conclusion for Classification Experiments

# Experiments with Neural Network.

$$
\begin{array}{|l|c|c|}
\hline
\textbf{Model} & \textbf{Mean Score} & \textbf{Standard Deviation} \\
\hline
\text{Logistic Regression} & 0.94796 & 0.04357 \\
\hline
\text{Support Vector Machines} & 0.96710 & 0.02934 \\
\hline
\text{k-Nearest Neighbors} & 0.92151 & 0.06043 \\
\hline
\text{Neural Networks} & 0.96108 & 0.04738 \\
\hline
\end{array}
$$


After training and evaluating the four classification models, all of them performed well against the dataset with the best overall performance being support vector machines with a mean score of around 0.97 and standard deviation of around 0.03.  Next was neural networks with a mean score of around 0.96 and standard deviation of around 0.05, followed by Logistic Regression and then k_Nearest Neighbors with a mean score of around 0.92 and standard deviation of around 0.06.  I believe support vector machines had the best performance as they are more versatile and robust than the rest.

## Results and Conclusion for Neural Network Experiments

In [50]:
df = pd.read_csv('ckd_feature_subset.csv') # Dataset loaded
X = df.drop('Target_ckd', axis=1)  
y = df['Target_ckd']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Prepare k-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Define different neural network configurations with comments
configs = [
    # Configuration 1: Basic network with one hidden layer and 5 neurons
    {"hidden_layer_sizes": (5,), "activation": "relu", "solver": "adam"},

    # Configuration 2: Shallow network with two hidden layers (10 and 5 neurons)
    {"hidden_layer_sizes": (10, 5), "activation": "tanh", "solver": "sgd", "learning_rate_init": 0.01},

    # Configuration 3: Deeper network with three hidden layers (10, 5, 3 neurons)
    {"hidden_layer_sizes": (10, 5, 3), "activation": "tanh", "solver": "adam", "max_iter": 1500},
]

# Evaluate the performance of each configuration
for i, config in enumerate(configs):
    # Specify max_iter separately to avoid conflicts if it's in the config dictionary
    max_iter = config.pop("max_iter", 1000)  # Provide a default if not in config
    model = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        activation=config["activation"],
        solver=config["solver"],
        max_iter=max_iter,  # Use the max_iter variable
        random_state=42,
        **{k: v for k, v in config.items() if k not in ["hidden_layer_sizes", "activation", "solver"]}
    )
    
    # Perform k-fold cross-validation
    mlp_scores = cross_val_score(model, X_scaled, y, cv=kf)
    
    # Compute mean and standard deviation
    mean_score = np.mean(mlp_scores)
    std_dev = np.std(mlp_scores)
    
    # Report results
    print(f"Configuration {i + 1}:")
    print(f"Hidden Layer Sizes: {config['hidden_layer_sizes']}")
    print(f"Activation Function: {config['activation']}")
    print(f"Solver: {config['solver']}")
    print(f"Cross-validation scores: {mlp_scores}")
    print(f"Mean score: {mean_score:.4f}, Standard deviation: {std_dev:.4f}")
    print("-" * 40)

c:\Users\<redacted>\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\<redacted>\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\<redacted>\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\<redacted>\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10

Configuration 1:
Hidden Layer Sizes: (5,)
Activation Function: relu
Solver: adam
Cross-validation scores: [0.87096774 0.93548387 0.83870968 0.93333333 0.96666667]
Mean score: 0.9090, Standard deviation: 0.0469
----------------------------------------
Configuration 2:
Hidden Layer Sizes: (10, 5)
Activation Function: tanh
Solver: sgd
Cross-validation scores: [0.93548387 1.         0.90322581 1.         0.96666667]
Mean score: 0.9611, Standard deviation: 0.0376
----------------------------------------
Configuration 3:
Hidden Layer Sizes: (10, 5, 3)
Activation Function: tanh
Solver: adam
Cross-validation scores: [0.96774194 1.         0.90322581 1.         0.96666667]
Mean score: 0.9675, Standard deviation: 0.0353
----------------------------------------


$$
\begin{array}{|l|c|c|}
\hline
\textbf{Model Configuration} & \textbf{Mean Score} & \textbf{Standard Deviation} \\
\hline
\text{Configuration 1: (5,)} & 0.9090 & 0.0469 \\
\hline
\text{Configuration 2: (10, 5)} & 0.9611 & 0.0376 \\
\hline
\text{Configuration 3: (10, 5, 3)} & 0.9675 & 0.0353 \\
\hline
\end{array}
$$

Out of the three configurations tested, the best performing was the deeper network with 3 hidden layers, which had a mean score of around 0.97 and a standard deviation of around 0.035.  Followed was the shallow network with 2 hidden layers, and then the basic network with 1 hidden layer.  However, even the most simple model performed very well with a mean score of around 0.91 and a standard deviation of 0.047.  From my testing, it appears that the number of hidden layers made the biggest difference in performance, with the activation function and solver having a lesser impact on the scores.